# 03 — Evaluation and robustness

**Reads notebook 02's handoff and turns it into the tables and the figure the manuscript
reports.**

## What this notebook consumes

Notebook 02 hands over five things, all produced by a fresh run of it:

| | |
|---|---|
| `regime_battery.csv` | per-seed metric rows, every regime |
| `regime_battery_summary.csv` | the across-seed summary |
| `yrbs_scores.parquet` | YRBS person-level predictions, twenty-six (regime, family) pairs |
| `loo_sensitivity_summary.csv` | leave-one-pillar-out, **per-seed rows despite the name** |
| `outcome_variants_summary.csv` | outcome variants, **per-seed rows despite the name** |

Section A validates that handoff before anything reads it.

## What the active code reads

**Only notebook 02's live handoff and the tracked specifications in `spec/`.** No active cell
reads a frozen published table, a working file with no producer, or a historical aggregate. Where
a section used to do that, it is deferred and says so.

**No table or figure is promoted into `outputs/` from a Restart-and-Run-All.** Everything
this notebook generates is written below the configured working root as a review candidate,
never inside the repository. The only file a Restart-and-Run-All writes there is the
hyperparameter fragment, which is built from the tracked specs alone and carries no
MCS-derived value. The full-grid fragment and the ventile figure are built as **candidates**, displayed and
left unwritten; everything else awaits your review.

## What is deferred, and why

Several sections of this notebook were built on archived inputs that a fresh run does not
produce. **Those sections are deferred rather than left in place**, because a frozen value displayed beside a live one reads as though the two came from the same run. The conformal coverage and prediction-set analysis is no longer among them — section K rebuilds it live from the score handoff, and only its archived cell-audit inputs stay deferred. The
YRBS subgroup analysis, the cohort-wide capacity view and the ventile analysis are no longer:
sections E, F and G rebuild all three live from the score handoff, at `>=1`, `>=2` and `>=3`.
Section H attaches evaluation-sample intervals to the cohort-wide quantities among them. Section L
lists what is still deferred.

This restructuring replaces those readers section by section. **Nothing below claims to be
reproducible from a fresh run unless it is**, and where a section still reads a frozen file it
says so at the point of reading.

## Disclosure

Notebook 02's battery files carry MCS-derived aggregate rows and are restricted by content.
Candidate tables computed here from them are restricted too, whatever shape they end up in.
**Structural validation is not disclosure approval**, and no approval follows from a directory,
a filename or a passing check. Only a complete table you have reviewed is promoted to the
manuscript outputs.

## Status vocabulary

Four different things, kept apart:

| | |
|---|---|
| `estimability_status` | whether the metric is mathematically defined for this cell |
| `stability_status` | the prespecified YRBS minimum-cell-size rule. **YRBS is open CDC data; this is an analytical rule, not disclosure** |
| `valid_fraction` | the share of bootstrap replicates in which a quantity was defined, on section H's evaluation-sample intervals. An interval is reported only at 99% or above |
| `disclosure_status` | MCS-derived values awaiting external-release review. **MCS only** |

## Metric conventions

**Both ROC-AUC and PR-AUC are reported.** They answer different questions and neither is a
substitute for the other.

- **ROC-AUC** measures how well the model ranks adolescents across all thresholds. It does not
  depend on prevalence, which is what makes it comparable to published comparators.
- **PR-AUC is prevalence-sensitive**, because its no-skill value *is* the prevalence. It must
  therefore **always be reported with the observed prevalence of the slice it was computed on**,
  and never compared across cohorts or across subgroup cells without accounting for their
  different nulls.
- **`>=3` may be relatively sparse.** How sparse is a question for the authorised run, not
  something asserted here — but where positives are few, PR-AUC and its uncertainty carry more
  of the information than ROC-AUC does, so both matter more at the higher-burden threshold
  rather than less.
- **`prauc_lift` is outside the approved reporting scope.** It is computed internally and enters
  no table and no prose.

In [5]:
import sys
from pathlib import Path

# src/ is a sibling of notebooks/; only one of these two exists from any
# working directory, and a path that does not exist is ignored.
sys.path[:0] = ["src", "../src"]

import numpy as np
import pandas as pd

import config
import evaluation
import inputs
import paper
import regime_names
import tex_tables


In [6]:
# CONFIGURATION — the scope of the DISPLAYS below. Section A's handoff validation does not
# read it: a canonical handoff is checked against its full declared scope whether or not this
# run is looking at one family.
FAMILIES   = regime_names.FAMILIES     # or ("CatBoost",) to look at one family
THRESHOLDS = (1, 2, 3)
SEEDS      = range(20)

# THIS IS THE INTERNAL NOTEBOOK, and it says so. Every diagnostic below stays here; a public
# copy is a SEPARATE file made by scripts/make_public_notebooks.py, which keeps output only in
# the cells spec/public_notebook_cells.json names and flips this flag in the copy alone.
PUBLIC_NOTEBOOK = True
print(f"displays scoped to {len(FAMILIES)} families x {len(THRESHOLDS)} thresholds "
      f"x {len(SEEDS)} seeds")

---
# A — The handoff from notebook 02

Everything below reads what a fresh notebook 02 run produced. This section checks that it is
complete and internally consistent before anything uses it.

**The declared score scope is twenty-six (regime, family) pairs**, and the two halves are
checked differently.

| | | |
|---|---|---|
| **non-recalibrated** | nine `unadapted`, nine `yrbs_local`, four raw focal pipelines = 22 pairs | **every one of the 1,320 cells must be score-bearing** |
| **recalibrated** | the same four focal pipelines after cross-fitted logistic recalibration = 4 pairs | each of the 240 cells is **either** score-bearing **or** explicitly non-estimable |

A recalibration cell whose k slice could not support three stratified folds has no mapping to
report; notebook 02 records it in `regime_battery.csv` with a `recal_status` and writes no
person-level rows for it. That is a declared outcome, not a gap, so the reconciliation is on
**keys**: every missing score cell must have exactly one matching non-estimable metric row at
the same regime, family, threshold and seed.

**The per-seed battery is what identifies those keys.** The across-seed summary cannot — it has
one row per (regime, family, threshold) and cannot say which seed was missing. Its
`recalibration_estimability_status` and seed counts are used only to cross-check the per-seed
reconciliation.

Nothing here prints a person-level row: counts, scope labels and pass/fail only.

In [7]:
# The canonical handoff scope. Independent of the display configuration above: narrowing
# FAMILIES narrows what is shown, never what a complete handoff must contain.
HANDOFF_FAMILIES = tuple(regime_names.FAMILIES)
HANDOFF_THRESHOLDS = (1, 2, 3)
HANDOFF_SEEDS = tuple(range(20))

# SOURCE: transfer.FOCAL_PIPELINES. Named here rather than imported, so this notebook does not
# pull in the modelling stack to read a contract. A drift fails loudly in the scope check below
# rather than passing quietly.
FOCAL_PIPELINES = (("target_only", "L1_LR"), ("fine_tune", "RF"),
                   ("fine_tune", "HistGB"), ("fine_tune", "CatBoost"))

DECLARED_PLAIN = ({("unadapted", f) for f in HANDOFF_FAMILIES}
                  | {("yrbs_local", f) for f in HANDOFF_FAMILIES}
                  | set(FOCAL_PIPELINES))
DECLARED_RECAL = {(f"{reg}_logistic_recal", fam) for reg, fam in FOCAL_PIPELINES}
DECLARED_SCOPE = DECLARED_PLAIN | DECLARED_RECAL

SCORE_COLUMNS = ["threshold", "family", "regime", "seed", "row_id", "y_true", "score"]
SCORE_KEY = ["threshold", "family", "regime", "seed", "row_id"]
CELL_KEY = ["regime", "family", "threshold", "seed"]

battery = pd.read_csv(inputs.resolve("regime_battery.csv"))
battery_summary = pd.read_csv(inputs.resolve("regime_battery_summary.csv"))
yrbs_scores = pd.read_parquet(Path(config.MCS_SCORES) / "yrbs_scores.parquet")

print(f"battery          {len(battery):,} per-seed rows, "
      f"{battery['regime'].nunique()} regimes")
print(f"battery summary  {len(battery_summary):,} across-seed rows")
print(f"scores           {len(yrbs_scores):,} person-level rows")

In [8]:
# ---- shape and scope -------------------------------------------------------
if list(yrbs_scores.columns) != SCORE_COLUMNS:
    raise ValueError(
        f"score columns are {list(yrbs_scores.columns)}, expected {SCORE_COLUMNS}")
if yrbs_scores.duplicated(SCORE_KEY).any():
    raise ValueError(
        f"{int(yrbs_scores.duplicated(SCORE_KEY).sum())} score rows share a scientific key")
if "mcs_internal" in set(yrbs_scores["regime"]):
    raise ValueError("an MCS-evaluated regime reached the YRBS score handoff")

# SENSITIVITY ONLY, AND CONDITIONAL. The nested per-split target redevelopment is not part
# of the recurring analysis: notebook 02 does not run it and does not emit its regime, so
# this branch does not fire on an ordinary run. It exists so that a separate sensitivity
# run's scores can be read without being reported as undeclared. It supplies no headline
# reference, no target-relative quantity and no publication row.
if "yrbs_resource_rich" in set(yrbs_scores["regime"]):
    DECLARED_SCOPE = set(DECLARED_SCOPE) | {("yrbs_resource_rich", family)
                                            for family in regime_names.FAMILIES}

_scope = set(map(tuple, yrbs_scores[["regime", "family"]].drop_duplicates().to_numpy()))
undeclared = _scope - DECLARED_SCOPE
if undeclared:
    raise ValueError(f"undeclared (regime, family) pair(s): {sorted(undeclared)}")
absent = DECLARED_SCOPE - _scope
if absent:
    raise ValueError(f"declared score pairs are missing from the handoff: {sorted(absent)}")

expected_thresholds = {f">={t}" for t in HANDOFF_THRESHOLDS}
actual_thresholds = set(yrbs_scores["threshold"])
if actual_thresholds != expected_thresholds:
    raise ValueError(f"thresholds present: {sorted(actual_thresholds)}; "
                     f"expected: {sorted(expected_thresholds)}")

print(f"scope       {len(_scope)} of {len(DECLARED_SCOPE)} declared (regime, family) pairs present")
print(f"key         unique on {SCORE_KEY}")
print(f"thresholds  {sorted(actual_thresholds)}")
print(f"seeds       {len(set(yrbs_scores['seed']))}")
print("mcs_internal absent from the handoff")


In [9]:
# ---- the non-recalibrated half: every declared cell must be score-bearing ----
scoring_keys = set(map(tuple, yrbs_scores[CELL_KEY].drop_duplicates().to_numpy()))
declared_plain_keys = {(r, f, f">={t}", s) for r, f in DECLARED_PLAIN
                       for t in HANDOFF_THRESHOLDS for s in HANDOFF_SEEDS}
missing_plain = declared_plain_keys - scoring_keys
if missing_plain:
    raise ValueError(
        f"{len(missing_plain)} non-recalibrated cell(s) carry no scores, e.g. "
        f"{sorted(missing_plain)[:3]}. Only a recalibration cell may be absent.")
print(f"non-recalibrated  {len(declared_plain_keys):,} declared cells, all score-bearing")

# ---- the recalibrated half ---------------------------------------------------
# ON KEYS, NOT COUNTS. A balanced duplication and omission satisfies a row count and leaves
# one cell attempted twice and another not at all, so the key set is what is checked.
RECAL_STATUSES = {"estimated", "single_class_slice", "too_few_minority_for_three_folds"}

declared_recal_keys = {(r, f, f">={t}", s) for r, f in DECLARED_RECAL
                       for t in HANDOFF_THRESHOLDS for s in HANDOFF_SEEDS}
recal_rows = battery[battery["regime"].isin({r for r, _ in DECLARED_RECAL})]

if recal_rows.duplicated(CELL_KEY).any():
    raise ValueError(
        f"{int(recal_rows.duplicated(CELL_KEY).sum())} recalibration cell(s) are attempted "
        f"more than once")
attempted_keys = set(map(tuple, recal_rows[CELL_KEY].to_numpy()))
if attempted_keys != declared_recal_keys:
    raise ValueError(
        f"attempted but not declared: {sorted(attempted_keys - declared_recal_keys)[:3]}; "
        f"declared but never attempted: {sorted(declared_recal_keys - attempted_keys)[:3]}")

_unknown = set(recal_rows["recal_status"]) - RECAL_STATUSES
if _unknown:
    raise ValueError(f"unrecognised recal_status value(s): {sorted(_unknown)}")

estimated_keys = set(map(tuple, recal_rows.loc[
    recal_rows["recal_status"] == "estimated", CELL_KEY].to_numpy()))
non_estimable_keys = attempted_keys - estimated_keys
recal_scoring_keys = scoring_keys & declared_recal_keys

# The two halves must line up cell for cell, in both directions.
if estimated_keys != recal_scoring_keys:
    raise ValueError(
        f"estimated but carrying no scores: {sorted(estimated_keys - recal_scoring_keys)[:3]}; "
        f"scored but not marked estimated: {sorted(recal_scoring_keys - estimated_keys)[:3]}")
if non_estimable_keys != declared_recal_keys - recal_scoring_keys:
    raise ValueError(
        "the non-estimable records and the missing score cells are not the same set")

print(f"recalibrated      {len(declared_recal_keys)} declared cells, each attempted once = "
      f"{len(estimated_keys)} estimated and score-bearing + "
      f"{len(non_estimable_keys)} non-estimable and absent")
if non_estimable_keys:
    print("  reasons:")
    for reason, n in (recal_rows.loc[recal_rows["recal_status"] != "estimated", "recal_status"]
                      .value_counts().items()):
        print(f"    {reason}: {n}")


In [10]:
# ---- the across-seed summary agrees with the per-seed statuses ---------------
_recal_summary = battery_summary[
    battery_summary["regime"].isin({r for r, _ in DECLARED_RECAL})].copy()
_summary_key = ["regime", "family", "threshold"]

if _recal_summary.duplicated(_summary_key).any():
    raise ValueError("a recalibration cell appears twice in the across-seed summary")
_declared_summary = {(r, f, f">={t}") for r, f in DECLARED_RECAL for t in HANDOFF_THRESHOLDS}
_have_summary = set(map(tuple, _recal_summary[_summary_key].to_numpy()))
if _have_summary != _declared_summary:
    raise ValueError(
        f"summary cells present but not declared: {sorted(_have_summary - _declared_summary)[:3]}; "
        f"declared but absent: {sorted(_declared_summary - _have_summary)[:3]}")

_status = set(_recal_summary["recalibration_estimability_status"])
if not _status <= {"complete", "incomplete"}:
    raise ValueError(
        f"unrecognised estimability status: {sorted(_status - {'complete', 'incomplete'})}")

_n = len(HANDOFF_SEEDS)
if not (_recal_summary["seeds_expected"] == _n).all():
    raise ValueError("seeds_expected is not the seed count")
if not (_recal_summary["seeds_attempted"] == _n).all():
    raise ValueError("a cell was not attempted on every seed")
if not (_recal_summary["seeds_estimated"] + _recal_summary["seeds_non_estimable"] == _n).all():
    raise ValueError("seeds_estimated + seeds_non_estimable does not account for every seed")


def non_estimable_reasons(statuses):
    """The summary's `reason=count` string, rebuilt from one cell's per-seed statuses."""
    counts = statuses[statuses != "estimated"].value_counts()
    return "; ".join(f"{reason}={n}" for reason, n in counts.items())


# Counts, and the reason strings behind them, reconciled against the per-seed rows.
_per_seed = (recal_rows.assign(_est=recal_rows["recal_status"] == "estimated")
             .groupby(_summary_key)
             .agg(est=("_est", "sum"), reasons=("recal_status", non_estimable_reasons))
             .reset_index())
_check = _recal_summary.merge(_per_seed, on=_summary_key, validate="one_to_one")
if not (_check["seeds_estimated"] == _check["est"]).all():
    raise ValueError("the summary's seed counts disagree with the per-seed statuses")
if not (_check["non_estimable_reasons"].fillna("") == _check["reasons"]).all():
    raise ValueError("the summary's non-estimable reasons disagree with the per-seed statuses")

_complete = _check["recalibration_estimability_status"] == "complete"
if not (_check.loc[_complete, "seeds_estimated"] == _n).all():
    raise ValueError("a cell marked complete does not carry every seed")

# Both directions: a complete cell has all twenty score keys, and an incomplete cell is
# missing exactly the seeds its per-seed rows say are non-estimable.
for _, r in _check.iterrows():
    key3 = (r["regime"], r["family"], r["threshold"])
    want = {(*key3, s) for s in HANDOFF_SEEDS}
    absent_seeds = want - scoring_keys
    expected_absent = {k for k in non_estimable_keys if k[:3] == key3}
    if absent_seeds != expected_absent:
        raise ValueError(
            f"{key3}: score seeds absent {sorted(s for *_, s in absent_seeds)}, "
            f"non-estimable records {sorted(s for *_, s in expected_absent)}")

# ---- an incomplete cell carries no ordinary across-seed result ----------------
# Derived from the frame's own columns rather than a list that would drift as metrics change.
# `prevalence` and `n_test` describe the evaluation slice and survive; the seed counts and the
# status describe estimability and must survive, or the row could not say why it is blank.
SLICE_AND_STATUS = {"prevalence", "n_test", "n_seeds", "seeds_expected", "seeds_attempted",
                    "seeds_estimated", "seeds_non_estimable",
                    "recalibration_estimability_status", "non_estimable_reasons",
                    "degenerate", "ft_mechanism"}
ORDINARY_RESULT_COLUMNS = [
    c for c in battery_summary.columns
    if c not in SLICE_AND_STATUS
    and (c.endswith(("_mean", "_sd", "_plo", "_phi"))
         or c in ("transfer_loss", "target_resource_gap", "adaptation_gain",
                  "target_gap_recovered"))]

_incomplete = _recal_summary[
    _recal_summary["recalibration_estimability_status"] == "incomplete"]
if len(_incomplete):
    _kept = _incomplete[ORDINARY_RESULT_COLUMNS].notna().any()
    if _kept.any():
        raise ValueError(
            f"incomplete recalibration cell(s) carry an ordinary result in "
            f"{sorted(_kept[_kept].index)}")

print(f"summary reconciles with the per-seed statuses on all {len(_check)} recalibration cells")
print(f"  {int(_complete.sum())} complete, {int((~_complete).sum())} incomplete")
print(f"  {len(ORDINARY_RESULT_COLUMNS)} ordinary result columns checked blank on "
      f"incomplete cells")
print("\nHANDOFF VALIDATED")


---
# B — Discrimination and calibration, per family

Across-seed means from the battery summary, projected to a long frame. Live: every value comes
from notebook 02's summary of this run.

The budget-lineage half of this section has been removed. It read
`per_family_calibration.csv`, a working file whose producer is an archived script — displaying
it beside the live s4 lineage would put two runs' numbers in one section.

In [11]:
# All three thresholds, passed explicitly — the helper's default is still two.
per_family = evaluation.per_family_calibration(
    lineage="s4", thresholds=HANDOFF_THRESHOLDS, families=FAMILIES)

print(f"{per_family.shape[0]} rows | metrics {sorted(per_family['metric'].unique())}")
display(per_family.groupby(["threshold", "metric"]).size().unstack("metric").fillna(0)
        .astype(int))

The two lineages **differ in shape** — the budget frame carries a `k` column and the battery
frame does not — which is why the function takes an explicit `lineage` and has no default.

**All three thresholds are passed explicitly, and the helper now requires them.** It used to
default to `>=1` and `>=2`, which meant an active caller could silently lose the third; the
argument has no default, so a threshold scope is always stated where it is chosen. PR-AUC carries its own
prevalence null on every row, and Brier, ECE, calibration intercept and calibration slope are
all present.

---
# C — The full transfer grid, at all three thresholds

Every family x regime x threshold cell the battery produced, in the tuned arm.

**`>=2` is primary; `>=1` and `>=3` are secondary.** All three are rendered. The threshold
sequence and the regime list are both **passed explicitly** below — neither is inferred from
whichever rows happen to be present, so a threshold that produced nothing reads as an empty
requested block rather than as a threshold nobody asked for.

**The regimes are declared by reporting group**, not by a fixed list inside the builder. The
two post-adaptation recalibration regimes are their own group: they are label-using, but they
are not members of the twelve-procedure mechanism comparison, and folding them in would change
what that comparison compares.

An **absent** cell and a **blank** one are different things. A (family, regime) the battery
never ran produces no row — the regime does not apply to that family. A row that exists with a
blank AUC is an incomplete recalibration cell: notebook 02 attempted it, could not estimate
every seed, and blanked the across-seed values. It stays, labelled `incomplete`.

In [12]:
# The scope, declared here rather than inside the builder.
GRID_THRESHOLDS = [f">={t}" for t in HANDOFF_THRESHOLDS]     # >=1, >=2, >=3
GRID_GROUPS = ["reference", "unadapted", "label_free", "label_using",
               "post_adaptation_recalibration"]
GRID_REGIMES = evaluation.reporting_regimes(GRID_GROUPS)

grid = evaluation.regime_grid(thresholds=GRID_THRESHOLDS, regimes=GRID_REGIMES,
                              families=FAMILIES)

print(f"{len(grid)} cells | {grid['family'].nunique()} families | "
      f"{grid['regime'].nunique()} of {len(GRID_REGIMES)} requested regimes | "
      f"thresholds {sorted(grid['threshold'].unique())}")
print(f"groups: {', '.join(GRID_GROUPS)}")

# Every requested threshold must be present and non-empty.
for t in GRID_THRESHOLDS:
    n = int((grid["threshold"] == t).sum())
    if not n:
        raise ValueError(f"{t} was requested and produced no row")
    print(f"  {t}: {n} cells")


**What the grid should contain.** The row count is not a number to remember: it is the
families in scope times the regimes the grid is built over times the thresholds it covers. The
cell above prints what this run produced, and the next cell states the expectation derived from
the declared vocabulary rather than from a previous run's output.

`_build_regime_grid` takes its thresholds from the caller and preserves their order, so all
three reach the grid and the coverage audit below says what each requested regime produced.

In [13]:
# What each requested regime produced. Nothing is manufactured to make the table rectangular:
# a (family, regime) the battery never ran simply shows a smaller family count.
grid_coverage = evaluation.regime_grid_coverage(
    grid, thresholds=GRID_THRESHOLDS, regimes=GRID_REGIMES, families=FAMILIES)
display(grid_coverage.set_index(["reporting_group", "regime"]))

_absent = grid_coverage[grid_coverage["cells"] == 0]
if len(_absent):
    print(f"{len(_absent)} requested regime(s) produced no cell at all: "
          f"{', '.join(_absent['regime'])}")
_inc = grid_coverage[grid_coverage["incomplete"] > 0]
if len(_inc):
    print(f"{int(_inc['incomplete'].sum())} incomplete cell(s), all recalibration: "
          f"blank means, still present and labelled")

print("\nRESTRICTED: this grid carries the MCS-derived source-reference column. It is a "
      "candidate for review, not an approved output.")

### Paired comparisons — descriptive only

**No significance claim is made anywhere in this notebook, and none is supported.** The twenty
splits are overlapping 75% draws from one sample: they share most of their training rows and
about a quarter of each test set, so the per-seed differences are not independent replicates and
the Wilcoxon signed-rank test's assumptions are not met.

Notebook 02 now retains the paired per-seed differences descriptively — the mean difference
across the twenty shared splits, with its across-split spread — and reports no p-value, no Holm
adjustment and no Hodges–Lehmann interval. The frozen `transfer_significance.csv` reader that
used to sit here has been removed rather than re-sourced.

A later phase will add **conditional** intervals for the comparisons whose person-level scores
are both in the handoff. Those describe evaluation-sample uncertainty with the fitted models
held fixed; they are not a test. **No significance dagger is currently supported.**

---
# D — Conformal prediction

**Reported in section I**, after the intervals, because it divides the same held-out evaluation
sets that section H builds and because the contrast with those intervals is the thing most worth
stating plainly: they are not the same kind of quantity.

---
# E — Subgroup performance across YRBS sex x ethnicity cells

Six questions, answered from the score handoff rather than from a frozen table.

1. How does discrimination vary across the eight YRBS sex x ethnicity cells?
2. Do calibration and outcome prevalence vary across those cells?
3. At a **system-wide** 10% and 20% review capacity — one ranking over that seed's whole
   evaluation set —
   which groups end up inside the flagged set, and where do false-positive rates differ?
4. Which cells are served best and worst, and how large is the gap between them?
5. Do the patterns hold at `>=1`, `>=2` and `>=3`?
6. How do the four focal raw pipelines compare with their recalibrated versions, where those
   are estimable?

**The cells** are `attr_sex` crossed with `attr_ethnicity_coarse` — male and female by White,
Black, Hispanic and Other — which is notebook 01's own vocabulary. No new crosswalk is built
and no category is merged. A respondent whose sex or ethnicity is missing falls in
`unclassified`. **They stay in the cohort-wide ranking** — they are part of the population the
service screens — and are excluded only from the eight-cell comparison. That exclusion is a
declared analytical choice and a limitation, not a neutral omission.

**The supported scope** is exactly what the handoff carries: nine `unadapted` families, nine
`yrbs_local` references, the four raw focal pipelines and their four recalibrated
counterparts. No subgroup result is manufactured for any other regime.

**The capacity cut is cohort-wide, and it is taken before any grouping.** A fixed review
capacity is a property of the service, not of a subgroup: at 10% the highest-scoring tenth of
everyone screened is reviewed, and the question is which groups fall inside that set. Ranking
within each cell instead would hand every group its own tenth and force every flag rate to 10%
by construction, which answers nothing. Subgroup flag rates here are **consequences of one
common ranking, not separately imposed quotas**, and they are free to differ.

**Capacities are 10% and 20%.** The 5% and 15% points belong to the label-budget curve and are
that experiment's operating points, not this one's.

**The minimum-cell-size rule is analytical, not disclosure.** YRBS is open CDC data. A cell with
fewer than fifty records in a seed is flagged `below_minimum` and its metrics are not reported,
because they cannot be estimated stably — the row stays, with its size, so the reader sees a
labelled gap rather than an absent group.

**MCS remains deferred.** MCS person-level scores are not persisted, so the three marginal
source-side panels cannot be rebuilt here and nothing below refits an MCS model. That boundary
is section L's.

In [14]:
# The one score handoff, and the attribute join. Both validate before returning: the loader
# checks the schema and the scientific key, the join checks that the attribute index is unique
# and that every row_id resolves exactly once.
scores = evaluation.load_yrbs_scores()
scored = evaluation.attach_subgroup_cells(scores)

SUPPORTED_REGIMES = sorted(set(scores["regime"]))
EXPECTED_CELLS = evaluation.SUBGROUP_CELLS          # eight, declared in src/evaluation.py

print(f"scores    {len(scores):,} rows | {len(SUPPORTED_REGIMES)} regimes | "
      f"{scores['family'].nunique()} families | "
      f"thresholds {sorted(scores['threshold'].unique())}")
print(f"join      {len(scored):,} rows, unchanged")
print(f"regimes   {', '.join(SUPPORTED_REGIMES)}")
print(f"capacities {[f'{int(c * 100)}%' for c in evaluation.CAPACITIES]} "
      f"(cohort-wide)  |  minimum cell size {evaluation.SUBGROUP_MIN_CELL_N}")

In [15]:
# MISSING ATTRIBUTES, AT RESPONDENT GRAIN. Each respondent appears once per regime, threshold
# and seed, so a count of unclassified *rows* is that repetition multiplied and says nothing
# about how many people lack an attribute.
_people = scored.drop_duplicates("row_id")[["row_id", "cell"]]
_n_people = len(_people)
_n_unclassified = int((_people["cell"] == evaluation.UNCLASSIFIED).sum())

print(f"respondents in the handoff        {_n_people:,}")
print(f"  with sex and ethnicity          {_n_people - _n_unclassified:,}")
print(f"  missing one or both             {_n_unclassified:,} "
      f"({100 * _n_unclassified / _n_people:.2f}%)")
print(f"score rows (the same people repeated across regime x threshold x seed) "
      f"{len(scored):,}")
print("\nUnclassified respondents REMAIN in the cohort-wide capacity ranking — they are part")
print("of the population screened. They are excluded only from the eight-cell comparison.")

# The eight cells are declared, not discovered. Every one must exist in the attribute frame,
# and nothing outside the vocabulary may appear.
_classified = set(_people["cell"]) - {evaluation.UNCLASSIFIED}
if not _classified <= set(EXPECTED_CELLS):
    raise ValueError(
        f"unexpected classified cell(s): {sorted(_classified - set(EXPECTED_CELLS))}")
_absent = set(EXPECTED_CELLS) - _classified
print(f"\ncells     {len(EXPECTED_CELLS)} declared: {', '.join(EXPECTED_CELLS)}")
if _absent:
    print(f"          {len(_absent)} not represented by any respondent: {sorted(_absent)}")
    print("          they remain visible non-estimable rows rather than disappearing")


### Per seed, before anything is averaged

Every metric is computed inside one seed's evaluation slice. Pooling a respondent's twenty
predictions and calling the result one sample would treat twenty correlated predictions as
twenty observations.

Each metric carries its own estimability. A cell with one outcome class leaves ROC-AUC and
PR-AUC undefined while the Brier score, the expected calibration error and the flag rate remain
perfectly well defined — one undefined metric does not blank the row.

PR-AUC carries the cell's own observed prevalence as its no-skill reference. `prauc_lift` is not
reported.

In [16]:
# THE COHORT-WIDE CUT IS TAKEN FIRST, per (regime, family, threshold, seed): the whole
# evaluable slice is ranked together — unclassified respondents included — the top 10% and 20%
# are selected, and only then are those flags grouped by cell. Discrimination and calibration
# still come from each cell's own rows.
#
# The grid is declared rather than discovered, so a (regime, family, threshold, seed) that
# produced no scores — a recalibration seed that could not be estimated — arrives as a visible
# non-estimable row instead of an absent one.
SUBGROUP_GRID = [(reg, fam, f">={t}", s)
                 for reg, fam in sorted(_scope)
                 for t in HANDOFF_THRESHOLDS for s in HANDOFF_SEEDS]

subgroup_per_seed = evaluation.subgroup_metrics_per_seed(scored, grid=SUBGROUP_GRID)

print(f"{len(subgroup_per_seed):,} rows | "
      f"{subgroup_per_seed['metric'].nunique()} metrics x {len(EXPECTED_CELLS)} cells x "
      f"{len(HANDOFF_SEEDS)} seeds x {len(SUBGROUP_GRID) // (3 * len(HANDOFF_SEEDS))} "
      f"regime-family pairs x 3 thresholds")
if set(subgroup_per_seed["cell"]) != set(EXPECTED_CELLS):
    raise ValueError("the eight-cell grid is incomplete")
print(f"metrics: {', '.join(sorted(subgroup_per_seed['metric'].unique()))}")
display(subgroup_per_seed.groupby(["estimability_status", "stability_status"])
        .size().rename("rows").reset_index())


### Across seeds, without survivor bias

The displayed point estimate is the mean of the twenty seed-specific values. **If any of the
twenty is missing or non-estimable, no mean is reported for that metric in that cell** — a mean
over whichever seeds happened to work is a different statistic from the one every other row
carries, and the two must not share a column. The reason is carried beside it, and the per-seed
rows above keep every value that was computed.

`split_sd` is the spread across the twenty overlapping splits. It describes **stability**, not
sampling uncertainty, and it is not a confidence interval. Conditional intervals are a later
phase.

In [17]:
# Notebook 02's per-seed recalibration statuses, so a recalibration cell that could not be
# estimated says why rather than merely coming up short.
_recal_status = battery.loc[
    battery["regime"].isin({r for r, _ in DECLARED_RECAL}),
    ["regime", "family", "threshold", "seed", "recal_status"]]

subgroup_summary = evaluation.summarise_subgroup_metrics(
    subgroup_per_seed, seeds_expected=len(HANDOFF_SEEDS), seed_status=_recal_status)

print(f"{len(subgroup_summary):,} (regime, family, threshold, cell, metric) rows")
display(subgroup_summary.groupby(["estimability_status", "stability_status"])
        .size().rename("cells").reset_index())

_inc = subgroup_summary[subgroup_summary["estimability_status"] == "incomplete"]
if len(_inc):
    print(f"\n{len(_inc)} incomplete cell(s) carry no mean. Reasons:")
    display(_inc.groupby(["metric", "estimability_reason"]).size().rename("cells")
            .reset_index().sort_values("cells", ascending=False).head(12))

### Discrimination and calibration, by cell

Unadapted transfer at the primary threshold, across the eight cells. A blank is a labelled gap:
either the metric was undefined in at least one seed, or the cell fell below the minimum size.

In [18]:
def subgroup_view(regime, threshold, metrics, *, families=None):
    """One regime and threshold, cells down and metrics across. Blanks are real."""
    sel = subgroup_summary[(subgroup_summary["regime"] == regime)
                           & (subgroup_summary["threshold"] == threshold)
                           & (subgroup_summary["metric"].isin(metrics))]
    if families is not None:
        sel = sel[sel["family"].isin(families)]
    return (sel.pivot_table(index=["family", "cell"], columns="metric", values="mean",
                            dropna=False)
               .reindex(columns=list(metrics)).round(4))


DISCRIMINATION = ("prevalence", "auc", "prauc", "brier", "ece", "cal_slope", "cal_intercept")
print("unadapted transfer at >=2 — mean over twenty seeds")
display(subgroup_view("unadapted", ">=2", DISCRIMINATION, families=FAMILIES))

### At a system-wide 10% and 20% review capacity

One ranking over that seed's whole evaluation set per model; the top 10% and 20% are
flagged; the cells below describe **who fell inside that shared set**.

- **flag rate** — the share of this cell that was flagged. It is *not* fixed at the capacity,
  and how far it departs is the point of the table.
- **precision** — of this cell's flagged respondents, how many have the outcome.
- **recall** — of this cell's positives, how many were flagged.
- **FPR** — of this cell's negatives, how many were flagged.

These are not a within-cell top 10% or 20%. No cell is given its own quota.

**A note on the operating point.** Ties at the capacity boundary are resolved deterministically using the order of respondents in that
seed's held-out set. The same rule is used for all compared models. This produces exactly the
stated capacity but remains an arbitrary allocation among equal scores. How often the rule
decides anything is reported in section H, from the tie diagnostic taken on every cut.

In [19]:
CAPACITY_METRICS = tuple(f"{stem}_at_{int(c * 100)}"
                         for c in evaluation.CAPACITIES
                         for stem in ("flagrate", "precision", "recall", "fpr"))

print(f"unadapted transfer at >=2 — cohort-wide "
      f"{', '.join(f'{int(c * 100)}%' for c in evaluation.CAPACITIES)} capacity, "
      f"mean over twenty seeds")
display(subgroup_view("unadapted", ">=2", CAPACITY_METRICS, families=FAMILIES))

# The cohort-wide capacity is preserved: the flagged counts across cells sum to the top-k.
_fr = subgroup_summary[(subgroup_summary["regime"] == "unadapted")
                       & (subgroup_summary["threshold"] == ">=2")
                       & (subgroup_summary["metric"] == "flagrate_at_10")]
print(f"\nflag rate at 10% ranges {_fr['mean'].min():.3f} to {_fr['mean'].max():.3f} across "
      f"cells and families — a spread, not a constant 0.10, because the cut is cohort-wide")

### Best-served and worst-served cells

The spread between the highest and lowest eligible cell. **A cell is eligible when its
across-seed mean is complete and it met the minimum-size rule in every seed.** If any cell of
the group is not eligible the gap is reported as **incomplete and no value is given** — a spread
computed over the surviving cells is systematically smaller than the one the question asks
about, and the excluded cells are usually the small ones carrying the disparity.

A flag-rate or FPR gap here is a real disparity in who the shared ranking selects. Under
the earlier, incorrect within-cell cut it would have been near zero by construction.

Uncertainty for the gap is a later phase.

In [20]:
subgroup_gaps = evaluation.subgroup_metric_gaps(subgroup_summary)

# THE DISPLAY NAMES ITS COLUMNS. `n_eligible` and `n_excluded` are counts of subgroup cells and
# of the respondents behind them, and a gap shown beside its cell counts is a gap whose cells
# can be sized. They stay in `subgroup_gaps` for internal reading and out of every display.
SUBGROUP_GAP_COLUMNS = ["family", "threshold", "metric", "max_cell", "max_value",
                        "min_cell", "min_value", "gap", "gap_status"]

print(f"{len(subgroup_gaps)} (regime, family, threshold, metric) groups")
display(subgroup_gaps.groupby(["gap_status"]).size().rename("groups").reset_index())

_g = subgroup_gaps[(subgroup_gaps["regime"] == "unadapted")
                   & (subgroup_gaps["metric"].isin(["auc", "prauc", "fpr_at_10", "fpr_at_20"]))]
print("\nunadapted transfer — highest minus lowest eligible cell")
display(_g[SUBGROUP_GAP_COLUMNS].round(4))

### Across the three thresholds, and raw against recalibrated

The same comparison at `>=1`, `>=2` and `>=3`, and the four focal pipelines beside their
recalibrated counterparts. A recalibrated cell whose mapping could not be fitted on every seed
shows no mean; its raw pipeline is **not** substituted for it.

In [21]:
print("AUC gap by threshold — unadapted transfer")
display(subgroup_gaps[(subgroup_gaps["regime"] == "unadapted")
                      & (subgroup_gaps["metric"] == "auc")]
        .pivot_table(index="family", columns="threshold", values="gap", dropna=False).round(4))

FOCAL_PAIRS = [("target_only", "L1_LR"), ("fine_tune", "RF"),
               ("fine_tune", "HistGB"), ("fine_tune", "CatBoost")]
_rows = [(f"{reg}{suffix}", fam) for reg, fam in FOCAL_PAIRS
         for suffix in ("", "_logistic_recal")]
_focal = subgroup_summary[
    subgroup_summary.set_index(["regime", "family"]).index.isin(_rows)
    & (subgroup_summary["metric"] == "auc") & (subgroup_summary["threshold"] == ">=2")]

print("\nsubgroup AUC at >=2 — the four focal pipelines, raw against recalibrated")
display(_focal.pivot_table(index=["family", "cell"], columns="regime", values="mean",
                           dropna=False).round(4))
_missing = _focal[_focal["mean"].isna()]
if len(_missing):
    print(f"{len(_missing)} focal cell(s) report no mean; the raw pipeline is not substituted.")
    display(_missing[["regime", "family", "cell", "seeds_estimated",
                      "estimability_reason", "stability_status"]])

### To read after the run

- Does the AUC gap between the best- and worst-served cells change much across the three
  thresholds, or is it stable?
- At the system-wide capacity, which cells are flagged more often than their share of the
  cohort, and which less? Do the cells that discriminate worst also carry the highest
  false-positive rate, or are those different cells?
- Does recalibration change the subgroup picture at all, or only the calibration metrics?
- How many cells fall below the minimum size, and at which threshold does that start to bite?

Nothing above answers these. They are questions for the authorised run.

**The MCS boundary.** MCS person-level scores are not persisted under the licence, so the three
marginal source-side panels the appendix asks for cannot be rebuilt from a handoff. Either
notebook 02 computes them in memory and emits aggregates, or they stay frozen and the manuscript
says so. Nothing here refits an MCS model or reads a frozen MCS panel.

---
# F — Cohort-wide screening within each held-out evaluation set

**Question.** At a system-wide 10% or 20% review capacity, what precision, recall, flag rate
and false-positive rate does each supported procedure achieve?

**Cohort-wide means across every group, not across every YRBS respondent.** Each model is
scored on the quarter of the cohort its seed held out, so one ranking is taken over that seed's
evaluation set — everybody in it, not one subgroup at a time — and repeated for each of the
twenty seeds. **The same cut section E groups by subgroup cell.** `cohort_capacity_flags` is called on the same rows with the same
rule, so the flagged set behind this table and the one behind the subgroup table are the same
respondents. The reconciliation is checked below rather than assumed.

`no_skill_precision` is the slice prevalence: what a random flag of the same size would
achieve, and the only thing that makes a precision at fixed capacity readable.

**Only 10% and 20%.** The 5% and 15% points belong to the label-budget curve and are that
experiment's operating points; nothing here defaults to them.

Computed per seed, then averaged over exactly twenty. If any seed is non-estimable, no ordinary
mean is reported for that metric — the counts and the reason are carried instead. The
across-seed SD is **split stability, not a confidence interval**.

**The tie caveat.** The cut always takes exactly `ceil(capacity x n)` rows.
Ties at the capacity boundary are resolved deterministically using the order of respondents in that
seed's held-out set. The same rule is used for all compared models. This produces exactly the
stated capacity but remains an arbitrary allocation among equal scores. Notebook 02's decile and quintile columns take
their selection from the same function, so the two agree by construction rather than by
coincidence; section H reports how often a boundary was actually tied.

In [22]:
cohort_op_per_seed = evaluation.cohort_operating_metrics_per_seed(
    scores, capacities=evaluation.CAPACITIES,
    grid=[(reg, fam, f">={t}", s) for reg, fam in sorted(_scope)
          for t in HANDOFF_THRESHOLDS for s in HANDOFF_SEEDS])

cohort_operating = evaluation.summarise_cohort_operating(
    cohort_op_per_seed, seeds_expected=len(HANDOFF_SEEDS), seed_status=_recal_status)

print(f"{len(cohort_operating):,} (regime, family, threshold, metric) rows")
_caps = sorted({m for m in cohort_operating["metric"] if "_at_" in m})
print(f"capacities: {', '.join(_caps)}")
_unapproved = [m for m in _caps if not m.endswith(("_at_10", "_at_20"))]
if _unapproved:
    raise ValueError(f"unapproved capacity metric(s) reached the table: {_unapproved}. "
                     f"5% and 15% belong to the label-budget curve, not to this one.")
display(cohort_operating.groupby(["estimability_status"]).size().rename("rows").reset_index())


In [23]:
def cohort_view(regime, threshold, metrics):
    sel = cohort_operating[(cohort_operating["regime"] == regime)
                           & (cohort_operating["threshold"] == threshold)
                           & (cohort_operating["metric"].isin(metrics))]
    return (sel.pivot_table(index="family", columns="metric", values="mean", dropna=False)
               .reindex(index=[f for f in FAMILIES if f in set(sel["family"])],
                        columns=list(metrics)).round(4))


OP_METRICS = ("prevalence", "precision_at_10", "recall_at_10", "flagrate_at_10", "fpr_at_10",
              "precision_at_20", "recall_at_20", "flagrate_at_20", "fpr_at_20")

for _t in [f">={t}" for t in HANDOFF_THRESHOLDS]:
    print(f"unadapted transfer at {_t} — cohort-wide, mean over twenty seeds")
    display(cohort_view("unadapted", _t, OP_METRICS))

### The two cuts reconcile, on every score cell

The cohort-wide flagged set and the per-cell tables must describe the same selection. That is
checked on **every score-bearing regime, family, threshold, seed and capacity**, not on one
probe.

**The audit uses raw counts, never a reportable rate.** A subgroup below the minimum-size rule
has its public precision, recall, flag rate and FPR blanked — but it still contains flagged
respondents, and reconstructing its count from a blanked rate would lose them from the total.
`capacity_accounting` therefore carries `n`, `n_flagged`, `tp`, `fp`, `fn` and `tn` per group,
separately from anything reportable, and the stability rule never touches them.

The order is: establish the cohort-wide cut, count, reconcile, *then* compute reportable rates,
*then* blank the small cells. A small cell cannot move the boundary, vanish from the accounting,
or be treated as having nothing flagged.

A recalibration seed with no score vector is not part of this audit — it has no capacity to
account for, and it is reconciled through its estimability status in section A.

These counts are **in-memory diagnostics, not manuscript outputs**. YRBS only; nothing here
touches an MCS quantity.

In [24]:
capacity_audit = evaluation.capacity_accounting(
    scored, capacities=evaluation.CAPACITIES,
    grid=[(reg, fam, f">={t}", s) for reg, fam in sorted(_scope)
          for t in HANDOFF_THRESHOLDS for s in HANDOFF_SEEDS])

capacity_reconciliation = evaluation.reconcile_capacity_accounting(capacity_audit)
_bad = capacity_reconciliation[~capacity_reconciliation["reconciles"]]

print(f"score cells checked   {len(capacity_reconciliation) // len(evaluation.CAPACITIES):,} "
      f"(regime x family x threshold x seed)")
print(f"capacities checked    {', '.join(f'{int(c * 100)}%' for c in evaluation.CAPACITIES)}")
print(f"rows checked          {len(capacity_reconciliation):,}")
print(f"disagreements         {len(_bad)}")
if len(_bad):
    display(_bad.groupby(["regime", "threshold", "capacity_pct"]).size().rename("rows")
            .reset_index())
    raise ValueError(
        f"{len(_bad)} score cell(s) do not reconcile: the cohort-wide flagged set and the "
        f"per-cell counts describe different selections")
print("\nclassified + unclassified = cohort, on n, flagged, TP, FP, FN and TN;")
print("cohort flagged = ceil(capacity x n_evaluable), on every checked cell.")


### To read after the run

- How far do precision and recall at 10% sit above the no-skill line, and does that hold at
  `>=3` where positives are scarcer?
- Does the ordering of families change between 10% and 20%, or between thresholds?
- Do the four focal pipelines buy anything at fixed capacity over unadapted transfer?

Nothing above answers these.

---
# G — Ventile stratification, live at all three thresholds

**Question.** Does observed outcome prevalence rise cleanly with predicted risk, and how far
apart are the top and bottom twentieths?

Computed from the score handoff, not from a published table. Two models — unadapted CatBoost
and the CatBoost target-trained reference — at `>=1`, `>=2` and `>=3`, over twenty seeds.

**The bars are held-out predictions; the reference line is the analytic cohort.** Each seed's
ventiles are formed from the quarter of the cohort that seed held out, so a bar describes the
adolescents that model actually scored — not a prediction for every YRBS respondent. The
horizontal line is outcome prevalence in the full threshold-specific analytic cohort, which is
the population those held-out sets are drawn from, so it is the right thing to read a bar
against.

**Binning is on the rank within one seed**, so every ventile holds the same number of
adolescents and the prevalences are comparable down the column. Ranking across seeds together
would pool a respondent's twenty predictions.

All twenty bins are emitted for every model and seed. An empty bin is a visible non-estimable
row, not a gap. A ventile whose prevalence is missing in any seed reports **no** ordinary
twenty-seed mean.

**The band is `mean ± 1.96 × split SD`, clipped to [0, 1].** It describes how the estimate
moves across twenty overlapping splits — **across-split stability, not a confidence interval**.
It is not a 95% band and implies no coverage. Conditional intervals are the next phase.

In [25]:
VENTILE_MODELS = [("unadapted", "CatBoost"), ("yrbs_local", "CatBoost")]
VENTILE_GRID = [(reg, fam, f">={t}", s) for reg, fam in VENTILE_MODELS
                for t in HANDOFF_THRESHOLDS for s in HANDOFF_SEEDS]

_v_scores = scores[scores.set_index(["regime", "family"]).index.isin(VENTILE_MODELS)]
ventile_per_seed = evaluation.ventile_prevalence_per_seed(_v_scores, grid=VENTILE_GRID)
ventiles = evaluation.summarise_ventiles(ventile_per_seed, seeds_expected=len(HANDOFF_SEEDS))

print(f"{len(ventile_per_seed):,} per-seed rows -> {len(ventiles)} summary rows")
for _t in [f">={t}" for t in HANDOFF_THRESHOLDS]:
    _n = ventiles[ventiles["threshold"] == _t]["ventile"].nunique()
    if _n != evaluation.VENTILES:
        raise ValueError(
            f"{_t} produced {_n} ventiles, expected {evaluation.VENTILES}")
    print(f"  {_t}: {_n} ventiles x {ventiles[ventiles.threshold == _t]['regime'].nunique()} models")
_inc = ventiles[ventiles["estimability_status"] == "incomplete"]
print(f"{len(_inc)} ventile(s) incomplete — no twenty-seed mean reported for them")


### The reference line: the analytic cohort the held-out sets are drawn from

The horizontal line is **outcome prevalence in the threshold-specific evaluable YRBS analytic
cohort** — the respondents with all five shared pillars observed, cut at that threshold.

It comes from the canonical pillar frame through the same strict outcome construction notebook
02 uses, so the definition and the evaluable universe match the score handoff by construction.
It is **not** an average over score rows: those repeat every respondent once per regime,
threshold and seed, so averaging them would weight by how many procedures happened to run. It is
also **not the mean prevalence of the test splits** — those are 25% draws, and their mean
prevalence is a different quantity from the analytic cohort's.

The hard-coded literals that used to sit here are gone, along with the unresolved note that
they disagreed with the published prevalences at the third decimal — the question does not
arise once the number is derived.

In [26]:
population_rate = evaluation.yrbs_analytic_prevalence(HANDOFF_THRESHOLDS)
display(population_rate)

# The evaluable universe behind the line must be the one the scores were computed on.
_score_n = (scores[scores["regime"] == "unadapted"]
            .groupby("threshold")["row_id"].nunique().rename("n_scored"))
_check = population_rate.set_index("threshold").join(_score_n)
print("\nanalytic cohort against the scored respondents, per threshold")
display(_check[["n_analytic", "n_scored", "prevalence"]])
print("The two need not be equal — the score frame carries the evaluable test rows of each "
      "split, the analytic cohort is every respondent with a complete outcome.")

### Top against bottom ventile

Two quantities, and they are **not interchangeable**.

- **Risk ratio** — top-ventile prevalence divided by bottom-ventile prevalence. It is the
  manuscript's claim, and it is **undefined when the bottom ventile's prevalence is exactly
  zero**. The historical implementation dropped such a seed on a truthiness test, which turned
  a twenty-seed mean into a mean over whichever seeds had a non-zero denominator. Here the seed
  is recorded as non-estimable with that reason and no ordinary mean is reported.
- **Risk difference** — top minus bottom. It stays defined when the bottom prevalence is zero.
  It is reported beside the ratio and **does not substitute for it**.

Uncertainty for either is deferred.

In [27]:
ventile_extremes = evaluation.summarise_metric_frame(
    evaluation.ventile_extremes_per_seed(ventile_per_seed),
    keys=["regime", "family", "threshold", "metric"], seeds_expected=len(HANDOFF_SEEDS))

display(ventile_extremes[["regime", "family", "threshold", "metric", "mean", "split_sd",
                          "seeds_estimated", "estimability_status", "estimability_reason"]]
        .set_index(["regime", "family", "threshold", "metric"]).round(4))

_rr = ventile_extremes[(ventile_extremes["metric"] == "risk_ratio_top_bottom")
                       & (ventile_extremes["estimability_status"] == "incomplete")]
if len(_rr):
    print(f"{len(_rr)} risk-ratio cell(s) are non-estimable; the risk difference for the same "
          f"cells remains defined and is shown above.")

### The figure, in memory only

Three panels — `>=1`, `>=2`, `>=3` — each showing unadapted CatBoost against the CatBoost
target-trained reference, the derived prevalence line, and the across-split stability bands.

**Nothing is written here.** The figure is displayed for inspection; it is saved further
down as a candidate, below the configured working root and never inside the repository, and
promotion into the manuscript happens only after you have reviewed the real-data result. A
ventile that could not be estimated in every seed is drawn as a gap, not interpolated.

**Layout note for the manuscript:** three panels need a full-width figure. The current draft
sizes this as a single-column two-panel figure, so the layout will need updating — recorded in
`docs/MANUSCRIPT_CHANGES.md`.

In [28]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

_thr = [f">={t}" for t in HANDOFF_THRESHOLDS]
_rate = population_rate.set_index("threshold")["prevalence"]
_STYLE = {"unadapted": dict(color="black", ls="-", marker="o", ms=2.6, label="unadapted transfer"),
          "yrbs_local": dict(color="0.35", ls="--", marker="^", ms=2.4, mfc="none",
                                label="target-trained reference")}

fig, axes = plt.subplots(1, len(_thr), figsize=(10.5, 3.1), sharey=True)
for ax, t in zip(np.atleast_1d(axes), _thr):
    for reg, style in _STYLE.items():
        d = ventiles[(ventiles["regime"] == reg) & (ventiles["threshold"] == t)
                     & (ventiles["family"] == "CatBoost")].sort_values("ventile")
        if d.empty:
            continue
        ax.plot(d["ventile"], d["mean"], lw=1.0, **style)
        ax.fill_between(d["ventile"], d["band_lo"], d["band_hi"], color=style["color"],
                        alpha=0.12, lw=0)
    ax.axhline(_rate.get(t, np.nan), color="0.55", ls=":", lw=0.8)
    ax.set_title(f"outcome {t}", fontsize=9)
    ax.set_xlabel("predicted-risk ventile")
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.set_ylim(0, 1)
    ax.yaxis.grid(True, color="0.92", lw=0.5); ax.set_axisbelow(True)
np.atleast_1d(axes)[0].set_ylabel("observed prevalence")
fig.legend(handles=[Line2D([0], [0], **{k: v for k, v in s.items() if k != "ms"})
                    for s in _STYLE.values()]
           + [Line2D([0], [0], color="0.55", ls=":", lw=0.8, label="analytic prevalence"),
              matplotlib.patches.Patch(color="0.5", alpha=0.12,
                                       label="mean +/- 1.96 x split SD (stability, not a CI)")],
           loc="lower center", ncol=4, frameon=False, fontsize=7, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.10, 1, 1])
plt.show()
print("displayed only — no figure written. Promotion follows your review of the real-data run.")

displayed only — no figure written. Promotion follows your review of the real-data run.


/var/folders/5x/3v4hf1wx1db08bghp4s514cr0000gn/T/ipykernel_38264/3833661833.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### To read after the run

- Does prevalence rise monotonically across the ventiles, or does it dip in the middle as the
  earlier run's curve did?
- How far apart are the top and bottom ventiles, and is the risk ratio estimable at `>=3`?
- Do the bands widen at `>=3`, where positives are scarcer?

Nothing above answers these.

---
# H — Uncertainty conditional on the fitted models

The variation across twenty seeds shows how sensitive the results are to the predefined
training and test splits, but it does not describe uncertainty in the adolescents observed in
those test sets. I therefore resample adolescents while keeping every fitted model and split
assignment fixed.

Notebook 02 cut the YRBS cohort 75/25 for each threshold and seed, stratified on the outcome,
and drew the k=500 adaptation sample from the training portion. The held-out quarter stayed
untouched until scoring, so each seed evaluates all comparable models on the same quarter and
nothing is scored in-sample. Test membership changes across seeds, so some adolescents appear in
several test sets. I draw one multiplicity for each eligible adolescent and reuse it in every
held-out set in which they appear. This preserves that overlap when the twenty seed-specific
results are averaged.

### Step 1 — the evaluation samples

In [29]:
yrbs_pillars = pd.read_parquet(config.YRBS_PILLARS)

eligible_rows, evaluation_sets, evaluation_design = evaluation.yrbs_evaluation_design(
    HANDOFF_THRESHOLDS, HANDOFF_SEEDS, pillars=yrbs_pillars)

display(evaluation_design)

### Step 2 — does the handoff match those samples?

Each model should carry scores for the adolescents its seed held out, all of them and once each,
with the outcome the cohort itself records. A model that is short, that carries someone else's
respondents, or that disagrees about an outcome is a broken handoff rather than a model
evaluated on a different population, so I stop rather than resample from it. Recalibration cells
that notebook 02 recorded as non-estimable have no rows to check and keep their reason.

In [30]:
coverage = evaluation.validate_score_coverage(
    scores, eligible_rows, evaluation_sets,
    declared=[(regime, family, f">={t}") for regime, family in sorted(DECLARED_SCOPE)
              for t in HANDOFF_THRESHOLDS],
    non_estimable=non_estimable_keys,
    reasons={tuple(row[:4]): row[4] for row in
             recal_rows.loc[recal_rows["recal_status"] != "estimated",
                            CELL_KEY + ["recal_status"]].to_numpy()},
    seeds=HANDOFF_SEEDS)

coverage_summary = coverage["summary"]
display(coverage_summary[["threshold", "eligible_respondents", "declared_cells", "complete",
                          "non_estimable", "invalid"]])

if len(coverage["problems"]):
    display(coverage["problems"].groupby(["regime", "family", "threshold", "problem"])
            .agg(model_seeds=("seed", "size"), expected=("expected_respondents", "max"),
                 worst_count=("count", "max")).reset_index())
    raise ValueError(
        f"{len(coverage['problems'])} model-seed(s) were not scored on the adolescents that "
        f"seed held out. Rerun notebook 02 or correct the recalibration record.")

not_estimable = coverage["states"].query("state == 'non_estimable'")
if len(not_estimable):
    print("recorded non-estimable before resampling, shown later without an interval:")
    display(not_estimable[["regime", "family", "threshold", "seeds_non_estimable", "reason"]])

### Step 3 — resample adolescents

Two thousand replicates are prespecified for the final analysis, and the seed makes the Monte
Carlo calculation reproducible. Two thousand is a clear prespecified count, not a proof of
convergence. If it turns out to be computationally infeasible I will revise the count before
looking at any interval endpoint — the timing cell below reports elapsed and projected runtime
only, and nothing statistical.

A replicate is invalid for a quantity when any of the twenty seeds leaves that quantity
undefined; it is dropped rather than repaired, and nothing is imputed or rebuilt from the seeds
that survived. Because undefinedness is not random — it happens where the quantity would have
been extreme — an interval built from what remains is conditional on the statistic being
defined. I therefore report an interval only when at least 99% of replicates are valid, and note
it whenever any were dropped. **The 99% is a conservative choice for this project, not a
statistical standard.**

In [31]:
import time

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 0
INTERVAL_ALPHA = 0.05
MIN_VALID_FRACTION = 0.99

TARGET_REFERENCE = "yrbs_local"   # every target-relative quantity is read against this
CHANGE_BASELINE = "unadapted"        # paired changes are measured from this
DISCRIMINATION = ("auc", "prauc", "brier")
CALIBRATION = ("ece", "cal_intercept", "cal_slope")
CAPACITY = tuple(m for m in evaluation.BOOTSTRAP_METRICS if "_at_" in m)

probe_model = next(iter(coverage["cells"]))
started = time.perf_counter()
evaluation.run_respondent_bootstrap(coverage, eligible_rows, replicates=2,
                                    seed=BOOTSTRAP_SEED, cells=[probe_model])
per_pass = (time.perf_counter() - started) / 3
print(f"timing only: {per_pass:.2f}s per model-pass, so roughly "
      f"{per_pass * len(coverage['cells']) * (BOOTSTRAP_REPLICATES + 1) / 3600:.1f} hours "
      f"for {len(coverage['cells'])} models x {BOOTSTRAP_REPLICATES} replicates")

bootstrap_results = evaluation.run_respondent_bootstrap(
    coverage, eligible_rows, replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED)

for threshold, draw in bootstrap_results["multiplicities"].items():
    drawn = draw.sum(axis=1)
    print(f"{threshold}  {draw.shape[0]} replicates x {draw.shape[1]:,} adolescents | "
          f"rows drawn {drawn.min():,}-{drawn.max():,} | largest multiplicity {int(draw.max())}")

### Step 4 — estimates and intervals

Counting every adolescent exactly once reproduces each seed's own held-out set, so the
unit-multiplicity calculation should return the figures already reported. The reconciliation
below checks that before anything is read.

`transfer_loss` measures the deterioration from the MCS home-cohort reference to unadapted
transfer. Its MCS side was evaluated inside MCS, and this notebook has no MCS person-level
scores, so the two cohorts cannot be resampled together and the row keeps its point estimate
without an interval.

In [33]:
# Values reported independently of the respondent bootstrap.

_reference = battery_summary["regime"].isin(["unadapted", "yrbs_local"])
_focal_pairs = set(FOCAL_PIPELINES) | set(DECLARED_RECAL)
_pairs = pd.MultiIndex.from_frame(battery_summary[["regime", "family"]])
_focal = (
    _pairs.isin(_focal_pairs)
    & pd.to_numeric(battery_summary["k"], errors="coerce").eq(500)
)

tuned = battery_summary[
    battery_summary["arm"].eq("tuned") & (_reference | _focal)
].copy()

_summary_key = ["regime", "family", "threshold"]
if tuned.duplicated(_summary_key).any():
    raise ValueError(
        "the fixed-configuration summary contains duplicate "
        "(regime, family, threshold) rows"
    )


# Performance values reported in Notebook 02.
performance_metrics = [
    metric
    for metric in evaluation.BOOTSTRAP_METRICS
    if "_at_" not in metric
]

reported_performance = tuned.melt(
    id_vars=_summary_key,
    value_vars=[f"{metric}_mean" for metric in performance_metrics],
    var_name="metric",
    value_name="mean",
)
reported_performance["metric"] = (
    reported_performance["metric"].str.removesuffix("_mean")
)
reported_performance["quantity"] = "performance"


# Capacity values calculated earlier in this notebook.
reported_capacity = (
    cohort_operating[
        cohort_operating["metric"].isin(CAPACITY)
    ][["regime", "family", "threshold", "metric", "mean"]]
    .copy()
)
reported_capacity["quantity"] = "performance"


# Procedure-level target-gap quantities reported in Notebook 02.
reported_target_gap = tuned.melt(
    id_vars=_summary_key,
    value_vars=["adaptation_gain", "target_gap_recovered"],
    var_name="metric",
    value_name="mean",
)
reported_target_gap["quantity"] = "target_gap"


# The target resource gap is defined once per family and threshold.
reported_cell_gap = (
    tuned[tuned["regime"].eq("unadapted")][
        ["family", "threshold", "target_resource_gap"]
    ]
    .rename(columns={"target_resource_gap": "mean"})
    .assign(
        regime="",
        metric="target_resource_gap",
        quantity="target_gap",
    )
)


reported_all = pd.concat(
    [
        reported_performance,
        reported_capacity,
        reported_target_gap,
        reported_cell_gap,
    ],
    ignore_index=True,
)


# Transfer loss remains a point estimate because Notebook 03 contains no
# person-level MCS predictions from which to construct an interval.
transfer_loss_points = (
    tuned[
        ["regime", "family", "threshold", "transfer_loss"]
    ]
    .rename(columns={"transfer_loss": "observed"})
    .drop_duplicates()
)


intervals, reconciliation = evaluation.bootstrap_intervals(
    bootstrap_results,
    alpha=INTERVAL_ALPHA,
    min_valid_fraction=MIN_VALID_FRACTION,
    reference=TARGET_REFERENCE,
    baseline=CHANGE_BASELINE,
    transfer_loss_points=transfer_loss_points,
    reported=reported_all,
    tolerance=5e-4,
)

display(reconciliation)

if reconciliation.empty or reconciliation["status"].iloc[0] != "ok":
    raise ValueError(
        "The bootstrap estimates differ from the independently reported "
        "values. Inspect the reconciliation before continuing."
    )

print("Bootstrap estimates reproduce the reported values at their stored precision.")

display(
    intervals.groupby(["quantity", "metric"])
    .size()
    .rename("interval_rows")
    .reset_index()
)

In [34]:
def interval_table(regime, threshold, metrics, quantity="performance"):
    """One regime and threshold as `estimate [lo, hi]` per family and metric."""
    chosen = intervals[(intervals["quantity"] == quantity)
                       & (intervals["regime"] == regime)
                       & (intervals["threshold"] == threshold)
                       & (intervals["metric"].isin(metrics))].copy()
    shown = []
    for _, row in chosen.iterrows():
        if row["observed"] != row["observed"]:
            shown.append("--")
        elif row["interval_lo"] != row["interval_lo"]:
            shown.append(f"{row['observed']:.4f}  (no interval)")
        else:
            shown.append(f"{row['observed']:.4f} "
                         f"[{row['interval_lo']:.4f}, {row['interval_hi']:.4f}]")
    chosen["shown"] = shown
    return (chosen.pivot_table(index="family", columns="metric", values="shown",
                               aggfunc="first", dropna=False)
                  .reindex(index=[f for f in FAMILIES if f in set(chosen["family"])],
                           columns=[m for m in metrics if m in set(chosen["metric"])]))


for threshold in [f">={t}" for t in HANDOFF_THRESHOLDS]:
    print(f"unadapted transfer at {threshold} — discrimination, Brier score and calibration")
    display(interval_table("unadapted", threshold, DISCRIMINATION + CALIBRATION))
    print(f"at a 10% and 20% review capacity")
    display(interval_table("unadapted", threshold, CAPACITY))
    # `transfer_loss` is the deterioration from the MCS home-cohort reference and is not
    # displayed: it is an MCS-derived quantity, and this block is one a public copy may
    # retain. It stays in `intervals` for internal reading.
    print("distance to the YRBS local reference")
    display(
        interval_table(
            "unadapted",
            threshold,
            ("distance_to_yrbs_local",),
            quantity="transfer",
        )
    )

    print("comparison with the YRBS local reference")
    display(
        interval_table(
            "unadapted",
            threshold,
            (
                "target_resource_gap",
                "adaptation_gain",
                "target_gap_recovered",
            ),
            quantity="target_gap",
        )
    )

In [36]:
# Change in AUC from unadapted transfer at >=2. The difference is taken
# within each seed on the same adolescents, then averaged across the twenty
# splits. These are descriptive intervals, not significance tests.
changes = intervals[
    (intervals["quantity"] == "paired_change")
    & (intervals["threshold"] == ">=2")
    & (intervals["metric"] == "auc")
].copy()

changes["shown"] = [
    (
        "--"
        if pd.isna(row["interval_lo"])
        else (
            f"{row['observed']:+.4f} "
            f"[{row['interval_lo']:+.4f}, {row['interval_hi']:+.4f}]"
        )
    )
    for _, row in changes.iterrows()
]

display(
    changes.pivot_table(
        index="family",
        columns="regime",
        values="shown",
        aggfunc="first",
        dropna=False,
    )
)


# Report how much of the bootstrap each interval uses and why an interval
# may have been withheld.
complete_intervals = int((intervals["valid_fraction"] == 1).sum())
print(
    f"{complete_intervals:,} of {len(intervals):,} quantities used "
    "every replicate"
)

withheld = intervals[
    intervals["observed"].notna()
    & intervals["interval_lo"].isna()
]
if len(withheld):
    print(f"{len(withheld):,} kept their estimate without an interval:")
    display(
        withheld.groupby(
            ["quantity", "metric", "interval_note"]
        )
        .agg(
            rows=("metric", "size"),
            lowest_valid_fraction=("valid_fraction", "min"),
        )
        .reset_index()
    )

partial = intervals[
    (intervals["valid_fraction"] < 1)
    & intervals["interval_lo"].notna()
]
if len(partial):
    print(f"{len(partial):,} are conditional on the statistic being defined:")
    display(
        partial.groupby(
            ["quantity", "metric", "invalid_reasons"]
        )
        .agg(
            rows=("metric", "size"),
            lowest_valid_fraction=("valid_fraction", "min"),
        )
        .reset_index()
    )


# The recovered share divides by the target resource gap. Its smallest
# replicate-level denominator shows whether the ratio approached instability.
target_gap_diagnostics = intervals.loc[
    intervals["metric"] == "target_gap_recovered",
    [
        "regime",
        "family",
        "threshold",
        "observed",
        "min_target_resource_gap",
        "valid_fraction",
    ],
].copy()

print(
    "\nsmallest target resource gap behind each recovered-share estimate:"
)
display(
    target_gap_diagnostics.sort_values(
        "min_target_resource_gap"
    ).head(8)
)

### Ties at the capacity boundary

The cut reviews exactly `ceil(capacity × n)` adolescents. Where several share the boundary score
the ordering is a stable sort, so ties are resolved by the order the rows already arrive in —
ascending respondent order within a seed — which is deterministic and reproducible rather than
arbitrary, but is not a considered policy either.

**If boundary ties are rare and small, the deterministic rule is retained and reported as a
limitation. If ties are frequent or large enough to affect conclusions, the tie policy will be
reconsidered before capacity results enter the manuscript.** The first authorised run is
therefore a diagnostic run for these numbers, not the final publication of capacity results.
Notebook 02's own capacity metrics are untouched in this pass; the diagnostic is what decides
whether a coordinated change to both notebooks is needed at all.

In [37]:
ties = bootstrap_results["capacity_ties"]
print(f"{len(ties):,} capacity cuts checked "
      f"({ties['regime'].nunique()} regimes x {len(HANDOFF_SEEDS)} seeds x "
      f"{ties['capacity'].nunique()} capacities)")
print(f"{int(ties['boundary_tied'].sum()):,} had a tied boundary score")
if ties["boundary_tied"].any():
    tied = ties[ties["boundary_tied"]].copy()
    tied["unplaced"] = tied["rows_at_boundary"] - tied["places_at_boundary"]
    print(f"largest tied group {int(tied['rows_at_boundary'].max())} rows | "
          f"largest excess over the places available {int(tied['unplaced'].max())}")
    display(tied.groupby(["threshold", "capacity"])
            .agg(cuts_with_a_tie=("boundary_tied", "size"),
                 largest_tied_group=("rows_at_boundary", "max"),
                 largest_excess=("unplaced", "max")).reset_index())

### What these intervals mean

They describe evaluation-sample uncertainty conditional on the existing models, splits and
adaptation samples. They do not include model-fitting uncertainty, alternative analysis choices,
the complex YRBS survey design or clustering — ignoring the design can understate uncertainty
where responses cluster within schools, so these intervals are narrower than a design-based
interval would be.

The across-seed spread reported elsewhere is a different quantity: it describes sensitivity to
the splits and the fits, not to who was surveyed.

The calibration slope can be unstable in small or nearly separated samples, and where it is, the
interval is wide. That is the result rather than a fault to correct, and nothing is trimmed or
capped.

No p-values, significance tests or multiplicity adjustments accompany any of this. Twenty
overlapping held-out sets are not independent replicates, and an interval that excludes zero is
not a test result.

Intervals for the subgroup metrics, the best-worst subgroup gap and the ventile bands each need
their construction rebuilt inside the replicate and are left for a later pass. `transfer_loss`
is retained as a point estimate, but its INTERVAL is unavailable because this notebook has no
MCS person-level predictions — the two cohorts cannot be resampled together. The conformal
quantities that need those same predictions are likewise unavailable rather than deferred.

---
# I — Conformal prediction

The bootstrap above describes uncertainty in performance measures. Conformal prediction answers
a different question: it constructs a prediction set for each adolescent and records whether that
set contains the observed outcome. The held-out evaluation set is divided once per threshold and
seed, with the same division used for every model being compared.

In [38]:
# The bootstrap estimates uncertainty in performance metrics. Conformal prediction instead
# constructs a prediction set for each adolescent. It uses the held-out scores already produced
# by Notebook 02 and does not refit any model or repeat the bootstrap.

CONFORMAL_ALPHA = 0.10
CONFORMAL_SPLIT_SEED = evaluation.CONFORMAL_SPLIT_SEED

conformal_scope = evaluation.conformal_models(FOCAL_PIPELINES)

# Carry forward the recorded reasons for recalibration cells that Notebook 02 could not estimate.
conformal_reasons = {
    tuple(row[:4]): row[4]
    for row in recal_rows.loc[
        recal_rows["recal_status"] != "estimated",
        CELL_KEY + ["recal_status"],
    ].to_numpy()
}

conformal_per_seed = evaluation.conformal_sets_per_seed(
    scores,
    evaluation_sets,
    models=conformal_scope,
    thresholds=HANDOFF_THRESHOLDS,
    seeds=HANDOFF_SEEDS,
    alpha=CONFORMAL_ALPHA,
    split_seed=CONFORMAL_SPLIT_SEED,
    non_estimable=non_estimable_keys,
    reasons=conformal_reasons,
)

conformal_summary = evaluation.summarise_conformal(
    conformal_per_seed,
    seeds_expected=len(HANDOFF_SEEDS),
)

conformal_status = (
    conformal_summary[
        [
            "regime",
            "family",
            "threshold",
            "seeds_expected",
            "seeds_estimated",
            "seeds_non_estimable",
            "status",
            "non_estimable_reasons",
        ]
    ]
    .drop_duplicates()
    .sort_values(["threshold", "regime", "family"])
)

print(
    f"{len(conformal_scope)} model cells evaluated across "
    f"{len(HANDOFF_THRESHOLDS)} thresholds and {len(HANDOFF_SEEDS)} seeds"
)

display(
    conformal_status.groupby(
        ["threshold", "status"],
        dropna=False,
    )
    .size()
    .rename("model_cells")
    .reset_index()
)

print("\nHeadline threshold: coverage and prediction-set composition")

display(
    conformal_summary[
        (conformal_summary["threshold"] == ">=2")
        & conformal_summary["metric"].isin(
            [
                "coverage",
                "singleton_share",
                "empty_share",
                "both_labels_share",
                "mean_set_size",
            ]
        )
    ][
        [
            "regime",
            "family",
            "metric",
            "mean",
            "split_sd",
            "seeds_estimated",
            "seeds_non_estimable",
            "status",
        ]
    ]
    .sort_values(["regime", "family", "metric"])
    .reset_index(drop=True)
)

---
# J — The LaTeX fragments

**One fragment is written; one is built as a candidate; one is deferred.**

| fragment | this phase |
|---|---|
| `hyperparameters.tex` | **written** — built entirely from `spec/local_model_settings.csv` and the live candidate-pool definitions in `models`, so it carries no MCS-derived value and no cross-validated score |
| the full transfer grid | **candidate only** — rendered in memory, validated structurally, not written. It carries the MCS-derived source-reference column |
| the subgroup performance table | **deferred** — its live producer is section E's frame; wiring it is a later step |
| the significance table | **removed** — the inference it rendered is not supported on overlapping splits, and nothing replaces it |

The renderer is now threshold-generic: it takes a threshold sequence and builds one AUC and one
PR-AUC column per threshold, so a third block needs no second implementation.

**Structural validity is not disclosure approval.** A candidate that renders correctly is a
candidate. Promotion is your explicit act after review, and no approval follows from a
directory, a filename or a passing check.

In [39]:
def _no_frozen_read(name):
    """The hyperparameter fragment builds from the tracked spec and reads nothing else.

    Passed in place of a reader so the claim is enforced rather than asserted: if the builder
    ever reaches for a working aggregate or a frozen table, this raises.
    """
    raise RuntimeError(
        f"the hyperparameter fragment must not read {name!r}: it is built from "
        f"spec/local_model_settings.csv and live code only")


tex_tables.report.clear()
tex_tables.build_appendix_tables(_no_frozen_read, paper.fragments_dir())

### The full-grid candidate

Rendered from this run's grid, validated for three threshold blocks, and left unwritten.

**Three states, distinguished.** `--` means the procedure does not apply to that family — the
battery never ran it. `NE` means the cell *was* attempted and could not be estimated on all
twenty seeds, so no across-seed value exists. A number is a number. Rendering the first two
identically would make an unestimable result indistinguishable from an inapplicable one, and the
candidate's footnote defines both. **No disclosure marker is introduced**: none has been
reviewed.

In [40]:
# The candidate's text, from the summary this run produced. Nothing is written.
_summary_for_grid = pd.read_csv(inputs.resolve("regime_battery_summary.csv"))


def this_runs_summary(_name):
    """The renderer's reader. It is handed this run's summary whatever table it asks for, so
    the candidate cannot pick up a frozen file."""
    return _summary_for_grid


fullgrid_lines, fullgrid_unavailable, fullgrid_rows = tex_tables.fullgrid_candidate(
    this_runs_summary, thresholds=GRID_THRESHOLDS)

_header = next(l for l in fullgrid_lines if "AUC" in l)
_missing_blocks = [t for t in GRID_THRESHOLDS if tex_tables.thr_disp(t) not in _header]
if _missing_blocks:
    raise ValueError(f"threshold block(s) {_missing_blocks} are missing from the candidate header")
print(f"candidate: {fullgrid_rows} data rows, {len(fullgrid_lines)} lines, "
      f"{len(fullgrid_unavailable)} unavailable")
print(f"threshold blocks: {', '.join(GRID_THRESHOLDS)}")
_ne = sum(l.count(tex_tables.NOT_ESTIMABLE) for l in fullgrid_lines)
print(f"cells marked NE (attempted, not estimable): {_ne}")

print("\nRESTRICTED — carries the MCS-derived source-reference column. Not written.")
print("Review the rendered table, then promote it deliberately.")


---
# K — Outcome robustness

Two live readers. Both are notebook 02's, produced by the same fresh run as everything in
section A.

**Both filenames say `summary` and both hold per-seed rows** — one row per variant per family
per seed, not an across-seed collapse. The names are notebook 02's and notebook 03 reads them
by name, so correcting them means changing producer and consumer together; it is recorded for
that pass rather than done here.

Scope is `>=1` for both, and deliberately so: a leave-one-out outcome at `>=2` over four pillars
is a different construct, and four of the six outcome variants are a single binary pillar, which
cannot reach two.

In [41]:
outcome_variants = pd.read_csv(inputs.resolve("outcome_variants_summary.csv"))
print(f"{len(outcome_variants)} per-seed rows, "
      f"{outcome_variants['outcome'].nunique()} outcome variants, "
      f"{outcome_variants['seed'].nunique()} seeds")
display(outcome_variants.pivot_table(index=["outcome", "model"], columns="role",
                                     values="auc", aggfunc="mean").round(4))

In [42]:
loo = pd.read_csv(inputs.resolve("loo_sensitivity_summary.csv"))
print(f"{len(loo)} per-seed rows, {loo['variant'].nunique()} variants, "
      f"{loo['seed'].nunique()} seeds")
display(loo.groupby(["variant", "dropped_pillar", "model"])
        [["mcs_auc_internal", "yrbs_auc_transfer", "transfer_gap"]]
        .agg(["mean", "std"]).round(4))

### Discrimination against calibration

The one comparison this notebook draws from the battery summary directly.

**To read after the run.** Compare the two rows of each block. Does the AUC range move much
between the source reference and unadapted transfer, and does the ECE or the calibration slope
move more? If discrimination is better preserved than calibration, that is a finding about this
run and belongs in the write-up as one; if it is not, the write-up has to say that instead.

Nothing above asserts an answer. The previous version of this cell printed one unconditionally,
which would have survived a run that did not support it.

---
# L — What this notebook does not yet do

Recorded plainly rather than left implied.

## Deferred, with no live producer today

| section | reads | producer |
|---|---|---|
| **conformal coverage and set composition** | **now live from `yrbs_scores.parquet`** — see section K | notebook 02 |
| conformal cell audit | `conformal_prereq_cells.csv`, `e7e_cell_counts_geq2.csv` | archived scripts |
| **YRBS subgroup performance** | **now live from `yrbs_scores.parquet`** — see section E | notebook 02 |
| MCS subgroup panels | `subgroup_panels_summary.csv` | archived script; MCS scores are not persisted |
| **cohort-wide capacity at 10% and 20%** | **now live** — section F, reconciled against section E's cut | notebook 02 |
| **ventile stratification** | **now live at all three thresholds** — section G, with a derived prevalence line | notebook 02 |
| threshold sensitivity, structure transfer, the calibration sweep | frozen published tables | archived scripts |
| the ventile figure | **built live in section G, displayed only** | promotion follows your review |
| intervals for the subgroup gap and the ventile bands | a later pass | section H covers the cohort-wide quantities |
| the full-grid fragment | **built live as a candidate in section J** | carries MCS-derived values; promotion follows your review |
| the subgroup fragment | section E's frame is sufficient; wiring it is a later step | deferred |

## Removed rather than deferred

The frozen significance reader and everything built on it; the nine-source consolidation and
its reconciliation; the per-family checkpointed subgroup batch and the diagnostic tree it wrote.
None of them is replaced by another file.

## Uncertainty

**Section H carries evaluation-sample intervals; nothing else does.** They resample
respondents while holding the fitted models, the training samples and seeds behind them, the
realised splits, preprocessing, adaptation and recalibration all fixed. They are **not
procedure-level confidence intervals**, they exclude uncertainty from model development and seed
selection, and they **ignore the survey design**, which can understate uncertainty. The across-split SD and the ventile band elsewhere describe how an estimate moves
across twenty overlapping splits — **stability**, not sampling uncertainty — and are not
intervals of any kind.

## Still to correct, in order

The order matters: the subgroup functions cannot be rebuilt while their score loader is still
blocked.

| phase | work | why it comes first |
|---|---|---|
| ~~2A~~ | ~~repoint the score loader~~ | **done** — `_score_sources` and the new `load_yrbs_scores` read one file |
| ~~2B~~ | ~~live YRBS subgroup calculation~~ | **done** — section E |
| **2C** | subgroup capacities at 10% and 20% | **corrected, not yet run.** Section E now takes one cohort-wide cut per model and seed and reports who fell inside it; an earlier version ranked within each cell, which forced every flag rate to the capacity. The corrected quantity has not been computed on real data |
| ~~3~~ | ~~three thresholds; cohort-wide capacity; live ventiles~~ | **done** — sections C, F and G. `_build_regime_grid` and the full-grid renderer are threshold-generic; every live consumer passes all three explicitly. **None of it has been run on real data** |
| ~~4A~~ | ~~evaluation-sample intervals — performance, capacity, the target-gap quantities, paired changes~~ | **done** — section H, at a prespecified 2,000 replicates. **Not run on real data** |
| **4B** | intervals for the subgroup metrics, the best-worst subgroup gap and the ventile bands | each needs a construction rebuilt inside the replicate — a max-minus-min over cells, and a rank binning |
| **later** | conformal; the MCS subgroup panels; wiring the subgroup fragment; the capacity score-tie rule; the survey-design limitation | separate decisions |

Outside section H, no uncertainty accompanies any number here except the across-split spread,
which is **stability, not a confidence interval**. `transfer_loss` and the conformal coverages will not
gain one at 4B either: both need MCS person-level predictions, which are excluded at source.

## The capacity estimand, corrected

The subgroup capacity metrics answer: *at a system-wide 10% or 20% review capacity, which
groups fall inside the flagged set?* One ranking is taken over that seed's whole evaluation set —
unclassified respondents included — and the resulting flags are then grouped by cell. An earlier
version of section E ranked within each cell, which gave every group its own tenth and made the
flag-rate gap approximately zero by construction. No conclusion about which group is flagged
more often is drawn here; that is for the authorised run.

## The capacity tie rule, still open

The 10% and 20% cuts are taken by `np.argsort`, which is not a stable sort, so respondents tied
at the boundary are ordered arbitrarily. Section E uses that behaviour unchanged so its numbers
agree with notebook 02's. It is not an approved tie policy, changing it would move published
values, and it is an open decision before final reporting.

## Disclosure

Nothing above constitutes disclosure approval. The battery files carry MCS-derived aggregates
and any candidate table built from them is restricted until you have reviewed the complete
table.

---
# M — Publication candidates

These files contain selected aggregate results from the completed analysis. They are working
publication candidates, not automatic disclosure clearance. The headline files contain the
primary threshold, while the threshold-sensitivity file retains all three outcome definitions.


In [43]:
import publication

MAIN_THRESHOLD = ">=2"


def at_main_threshold(frame, name):
    """Select the primary outcome threshold and stop if its scope is wrong."""
    selected = frame[frame["threshold"] == MAIN_THRESHOLD].copy()
    return publication.require_primary_threshold(
        selected,
        MAIN_THRESHOLD,
        name,
    )


# Cohort-wide performance at fixed review capacities.
publication.save_table(
    at_main_threshold(cohort_operating, "capacity_results"),
    "capacity_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "metric",
        "mean",
        "split_sd",
        "seeds_estimated",
        "estimability_status",
    ],
)

# Eight-cell sex-by-ethnicity evaluation.
publication.save_table(
    at_main_threshold(subgroup_summary, "subgroup_results"),
    "subgroup_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "cell",
        "metric",
        "mean",
        "split_sd",
        "seeds_estimated",
        "estimability_status",
        "stability_status",
    ],
)

# Respondent-bootstrap intervals conditional on the fitted models and realised splits.
publication.save_table(
    at_main_threshold(intervals, "uncertainty_intervals"),
    "uncertainty_intervals.csv",
    [
        "quantity",
        "regime",
        "family",
        "threshold",
        "metric",
        "baseline",
        "observed",
        "interval_lo",
        "interval_hi",
        "valid_fraction",
        "interval_note",
    ],
)

# Split-conformal coverage and prediction-set behaviour.
publication.save_table(
    at_main_threshold(conformal_summary, "conformal_results"),
    "conformal_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "metric",
        "mean",
        "split_sd",
        "seeds_expected",
        "seeds_estimated",
        "seeds_non_estimable",
        "status",
        "seeds_with_clipped_quantile",
        "non_estimable_reasons",
    ],
)

# All three outcome thresholds are retained only in the sensitivity file.
threshold_sensitivity = publication.add_threshold_roles(
    battery_summary[battery_summary["arm"] == "tuned"].copy(),
    keys=("regime", "family"),
)

publication.save_table(
    threshold_sensitivity,
    "threshold_sensitivity.csv",
    [
        "regime",
        "family",
        "threshold",
        "threshold_role",
        "auc_mean",
        "auc_sd",
        "prauc_mean",
        "prevalence",
        "ece_mean",
        "cal_slope_mean",
        "cal_intercept_mean",
        "target_resource_gap",
        "target_gap_recovered",
    ],
)

# NO PER-SEED SELECTION STATUS IS CARRIED. The headline target reference takes one fixed
# configuration from the tracked specification, so every split is configured by
# construction and there is no per-seed search outcome to report. The nested per-split
# sensitivity keeps its own status on its own frame and supplies nothing here.
target_gap_candidates = at_main_threshold(
    battery_summary[battery_summary["arm"] == "tuned"].copy(),
    "target_gap_results",
)

if target_gap_candidates["target_resource_gap"].isna().any():
    missing_families = sorted(
        target_gap_candidates.loc[
            target_gap_candidates["target_resource_gap"].isna(),
            "family",
        ]
        .astype(str)
        .unique()
    )
    raise ValueError(
        "No target resource gap could be computed for: "
        f"{missing_families}"
    )

publication.save_table(
    target_gap_candidates,
    "target_gap_results.csv",
    [
        "regime",
        "family",
        "threshold",
        "auc_mean",
        "target_resource_gap",
        "adaptation_gain",
        "target_gap_recovered",
        "target_gap_reason",
    ],
)

# The ventile figure was constructed earlier in this notebook.
publication.save_figure(
    fig,
    "ventile_prevalence.png",
)

print("Notebook 3 publication candidates have been replaced from the current run.")